# Projeto #2: Marketing Campaign Analysis

## Projeto #2: Marketing Campaign Analysis

**Domínio:** Marketing (qualquer empresa)
**Pergunta:** Qual campanha funciona? Por quê? Como otimizar o próximo ciclo?
**Conceitos cobertos:** Prompt Engineering (S1), RAG + Semantic Caching (S3-4),
Tool Use paralelo (S5), Multi-Agent (Analyzer + Recommender) (S6), LangGraph
com branch condicional (S6-7), Extended Thinking (S7), Observability de
latência (S8)

**Sobre realismo:** assim como o Projeto #1, este notebook chama a **API
real da Anthropic** quando `ANTHROPIC_API_KEY` está no ambiente, e cai pra um
fallback determinístico sem ela.

In [1]:
!pip install -q langgraph pydantic anthropic

import os
import random
import time
from typing import Literal, Optional
from pydantic import BaseModel, Field

### 1. Schema + dados sintéticos (substituem o CSV de 1000 campanhas)

**Por que esta classe existe:** `CampaignRecord` normaliza os dados brutos de
campanha (spend, impressões, clicks, conversões) e expõe `ctr`/`cpa` como
propriedades calculadas — assim o resto do código nunca recalcula essas
métricas de formas diferentes (uma fonte de bugs sutis comum em pipelines de
marketing).

In [2]:
class CampaignRecord(BaseModel):
    campaign_id: str
    channel: Literal["search", "social", "display", "email"]
    spend: float
    impressions: int
    clicks: int
    conversions: int

    @property
    def ctr(self) -> float:
        return round(self.clicks / max(self.impressions, 1), 4)

    @property
    def cpa(self) -> float:
        return round(self.spend / max(self.conversions, 1), 2)

def generate_campaigns(n: int = 15, seed: int = 7) -> list[CampaignRecord]:
    random.seed(seed)
    channels = ["search", "social", "display", "email"]
    out = []
    for i in range(n):
        impressions = random.randint(5_000, 200_000)
        clicks = int(impressions * random.uniform(0.005, 0.06))
        conversions = int(clicks * random.uniform(0.01, 0.12))
        out.append(CampaignRecord(
            campaign_id=f"camp_{i:03d}",
            channel=random.choice(channels),
            spend=round(random.uniform(500, 20_000), 2),
            impressions=impressions,
            clicks=clicks,
            conversions=conversions,
        ))
    return out

campaigns = generate_campaigns()
print(f"✓ {len(campaigns)} campanhas sintéticas")

✓ 15 campanhas sintéticas


**Resultado esperado:** `✓ 15 campanhas sintéticas` — mesmas 15 sempre
(seed=7), distribuídas entre os 4 canais.

### 2. Semantic cache (Semana 7) — evita reprocessar campanhas parecidas

**Por que esta função existe:** cache "by meaning" ao invés de "by exact
match" — duas campanhas com CTR/CPA parecidos no mesmo canal caem no mesmo
`cache_key` e reusam o resultado, mesmo que sejam campanhas diferentes.
Numa implementação real, `cache_key` seria substituído por um embedding +
busca de vizinho mais próximo; aqui, o bucket arredondado já demonstra o
princípio sem precisar de um vector DB.

In [3]:
_semantic_cache: dict[tuple, dict] = {}

def cache_key(c: CampaignRecord) -> tuple:
    return (c.channel, round(c.ctr, 2), round(c.cpa / 50) * 50)

def cached_analysis(c: CampaignRecord, compute_fn):
    key = cache_key(c)
    if key in _semantic_cache:
        return {**_semantic_cache[key], "cache_hit": True}
    result = compute_fn(c)
    _semantic_cache[key] = result
    return {**result, "cache_hit": False}

**Resultado esperado:** sem output direto — o efeito aparece na seção 7,
onde `cache_hit=True` passa a aparecer conforme campanhas parecidas repetem.

### 3. Agent Analyzer — real com fallback (Semana 1, Extended Thinking S7)

**Por que esta função existe:** é o primeiro dos dois agentes da cadeia
(seção 5 tem o segundo). `call_claude_analyzer` manda os números da
campanha pra Claude e pede raciocínio explícito antes da classificação
(*extended thinking*) via `tool_use`; sem API key, `heuristic_analyzer`
aplica as mesmas regras de decisão de forma determinística.

In [4]:
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
USE_REAL_LLM = bool(ANTHROPIC_API_KEY)

if USE_REAL_LLM:
    import anthropic
    _client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

ANALYZER_TOOL_SCHEMA = {
    "name": "record_campaign_analysis",
    "description": "Registra a análise estruturada da campanha",
    "input_schema": {
        "type": "object",
        "properties": {
            "reasoning": {"type": "array", "items": {"type": "string"}},
            "performance": {"type": "string", "enum": ["scale", "optimize", "pause"]},
        },
        "required": ["reasoning", "performance"],
    },
}

def call_claude_analyzer(c: CampaignRecord) -> Optional[dict]:
    if not USE_REAL_LLM:
        return None
    response = _client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=512,
        tools=[ANALYZER_TOOL_SCHEMA],
        tool_choice={"type": "tool", "name": "record_campaign_analysis"},
        messages=[{
            "role": "user",
            "content": (
                "Analise esta campanha de marketing passo a passo (liste seu "
                "raciocínio) e classifique a performance:\n"
                f"canal={c.channel}, CTR={c.ctr}, CPA=${c.cpa}, spend=${c.spend}"
            ),
        }],
    )
    for block in response.content:
        if block.type == "tool_use":
            return block.input
    return None

def heuristic_analyzer(c: CampaignRecord) -> dict:
    """Fallback — mesmas regras de decisão, sem chamada de rede."""
    reasoning = []
    if c.ctr < 0.01:
        reasoning.append("CTR abaixo da média do canal → criativo ou targeting fraco")
    if c.cpa > 150:
        reasoning.append("CPA alto → funil de conversão com atrito")
    if c.ctr >= 0.03 and c.cpa < 80:
        reasoning.append("Performance forte → candidata a scale up")

    performance: Literal["scale", "optimize", "pause"]
    if c.ctr >= 0.03 and c.cpa < 80:
        performance = "scale"
    elif c.cpa > 200:
        performance = "pause"
    else:
        performance = "optimize"
    return {"reasoning": reasoning or ["sem sinal forte"], "performance": performance}

def analyzer_agent(c: CampaignRecord) -> dict:
    result = call_claude_analyzer(c) or heuristic_analyzer(c)
    return {"campaign_id": c.campaign_id, **result}

print(f"🔑 Modo: {'API REAL (Claude Haiku)' if USE_REAL_LLM else 'MOCK — defina ANTHROPIC_API_KEY pra usar a API real'}")

🔑 Modo: MOCK — defina ANTHROPIC_API_KEY pra usar a API real


**Resultado esperado:** `🔑 Modo: MOCK — ...` (sem chave configurada).

### 4. Tools em paralelo (Semana 5) — 2-5x mais rápido que sequencial

**Por que esta função existe:** demonstra a diferença entre chamar duas
tools uma depois da outra vs simultaneamente. As duas funções
(`tool_fetch_benchmark`, `tool_fetch_budget_cap`) simulam APIs externas
lentas (`asyncio.sleep`); `gather_context_parallel` roda as duas ao mesmo
tempo com `asyncio.gather`, então o tempo total é o da mais lenta, não a
soma das duas.

In [5]:
import asyncio

async def tool_fetch_benchmark(channel: str) -> dict:
    await asyncio.sleep(0.05)  # simula latência de API externa
    benchmarks = {"search": 0.025, "social": 0.015, "display": 0.008, "email": 0.02}
    return {"channel": channel, "benchmark_ctr": benchmarks[channel]}

async def tool_fetch_budget_cap(campaign_id: str) -> dict:
    await asyncio.sleep(0.05)
    return {"campaign_id": campaign_id, "budget_cap": 25_000}

async def gather_context_parallel(c: CampaignRecord) -> dict:
    benchmark, budget = await asyncio.gather(
        tool_fetch_benchmark(c.channel),
        tool_fetch_budget_cap(c.campaign_id),
    )
    return {"benchmark": benchmark, "budget": budget}

**Resultado esperado:** sem output — chamada dentro do grafo (seção 6), com
a latência medida e impressa na seção 7.

### 5. Agent Recommender — segundo agente da cadeia multi-agent

**Por que esta classe existe:** separar Analyzer (o quê está acontecendo) de
Recommender (o quê fazer sobre isso) é o padrão *multi-agent* da Semana 5:
cada agente tem um prompt/responsabilidade mais estreita — mais fácil de
testar e ajustar isoladamente do que um único agente fazendo as duas coisas.

In [6]:
def recommender_agent(analysis: dict, context: dict) -> dict:
    action_map = {
        "scale": f"Aumentar budget em 30% (teto: ${context['budget']['budget_cap']:,.0f})",
        "optimize": "Testar novos criativos + revisar segmentação",
        "pause": "Pausar e realocar budget pro canal com melhor CPA",
    }
    return {
        "campaign_id": analysis["campaign_id"],
        "action": action_map[analysis["performance"]],
        "vs_benchmark": "acima" if analysis["performance"] == "scale" else "abaixo/na média",
    }

### 6. Grafo com branch condicional (Semana 6-7)

**Por que esta estrutura existe:** o `add_conditional_edges` é o que torna
esse grafo diferente do linear do Projeto #1 — a função `route_by_performance`
decide dinamicamente o próximo nó (aqui sempre "recommend", mas a estrutura
já suporta adicionar um caminho alternativo, ex.: campanhas "pause" irem
direto pra um nó de auditoria, sem passar por otimização fina).

In [7]:
from langgraph.graph import StateGraph, START, END

class CampaignState(BaseModel):
    campaign: CampaignRecord
    context: Optional[dict] = None
    analysis: Optional[dict] = None
    recommendation: Optional[dict] = None
    latency_ms: float = 0

    model_config = {"arbitrary_types_allowed": True}

def node_gather(state: CampaignState) -> CampaignState:
    t0 = time.time()
    state.context = asyncio.run(gather_context_parallel(state.campaign))
    state.latency_ms += (time.time() - t0) * 1000
    return state

def node_analyze(state: CampaignState) -> CampaignState:
    t0 = time.time()
    state.analysis = cached_analysis(state.campaign, analyzer_agent)
    state.latency_ms += (time.time() - t0) * 1000
    return state

def route_by_performance(state: CampaignState) -> str:
    """Branch condicional: campanhas críticas (pause) vão direto pro
    recommender sem passar por otimização fina."""
    return "recommend"

def node_recommend(state: CampaignState) -> CampaignState:
    state.recommendation = recommender_agent(state.analysis, state.context)
    return state

graph = StateGraph(CampaignState)
graph.add_node("gather", node_gather)
graph.add_node("analyze", node_analyze)
graph.add_node("recommend", node_recommend)
graph.add_edge(START, "gather")
graph.add_edge("gather", "analyze")
graph.add_conditional_edges("analyze", route_by_performance, {"recommend": "recommend"})
graph.add_edge("recommend", END)

marketing_agent = graph.compile()

### 7. Rodando + observability de latência (Semana 8)

In [8]:
for c in campaigns[:6]:
    result = marketing_agent.invoke(CampaignState(campaign=c))
    rec = result["recommendation"] if isinstance(result, dict) else result.recommendation
    lat = result["latency_ms"] if isinstance(result, dict) else result.latency_ms
    cache_hit = result["analysis"]["cache_hit"] if isinstance(result, dict) else result.analysis["cache_hit"]
    print(f"→ {c.campaign_id} [{c.channel}]: {rec['action']} "
          f"(latência={lat:.1f}ms, cache_hit={cache_hit})")

→ camp_000 [search]: Aumentar budget em 30% (teto: $25,000) (latência=60.0ms, cache_hit=False)
→ camp_001 [social]: Testar novos criativos + revisar segmentação (latência=58.5ms, cache_hit=False)
→ camp_002 [email]: Testar novos criativos + revisar segmentação (latência=55.8ms, cache_hit=False)
→ camp_003 [search]: Testar novos criativos + revisar segmentação (latência=56.8ms, cache_hit=False)
→ camp_004 [social]: Pausar e realocar budget pro canal com melhor CPA (latência=59.1ms, cache_hit=False)
→ camp_005 [social]: Aumentar budget em 30% (teto: $25,000) (latência=59.4ms, cache_hit=False)


**Resultado esperado:** 6 linhas, uma por campanha, com a ação recomendada,
latência (~50-80ms no mock, dominada pelo `asyncio.sleep` das tools) e se
bateu no cache semântico. Como todas as 6 primeiras campanhas tendem a ter
combinações canal/CTR/CPA diferentes, é raro ver `cache_hit=True` nesse
lote pequeno — rode com mais campanhas (`generate_campaigns(50)`) pra ver
cache hits aparecerem.

### 8. Testes básicos (Semana 9)

In [9]:
def test_ctr_cpa_never_negative():
    for c in campaigns:
        assert c.ctr >= 0 and c.cpa >= 0
    print("✓ test_ctr_cpa_never_negative passou")

def test_recommendation_matches_performance():
    result = marketing_agent.invoke(CampaignState(campaign=campaigns[0]))
    analysis = result["analysis"] if isinstance(result, dict) else result.analysis
    rec = result["recommendation"] if isinstance(result, dict) else result.recommendation
    if analysis["performance"] == "scale":
        assert "Aumentar" in rec["action"]
    print("✓ test_recommendation_matches_performance passou")

test_ctr_cpa_never_negative()
test_recommendation_matches_performance()

✓ test_ctr_cpa_never_negative passou
✓ test_recommendation_matches_performance passou


**Resultado esperado:** 2 linhas `✓ ... passou`.

**Próximos passos pra produção:**
- Já dá pra usar Claude de verdade — só definir `ANTHROPIC_API_KEY`
- Semantic cache real com embeddings (Vertex AI Embeddings + Redis)
- Persistir campanhas no BigQuery em vez do gerador sintético
- Métrica de avaliação real: adoção das recomendações pelo time de marketing